In [1]:
#  Mount Drive and verify GPU and data
from google.colab import drive
drive.mount('/content/drive')

import torch
import os

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

# Verify CTR-GCN format data exists on Drive
ctrgcn_dir = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/ctrgcn_format'
npz_path = os.path.join(ctrgcn_dir, 'HRI30_CS.npz')

if os.path.exists(npz_path):
    size_mb = os.path.getsize(npz_path) / 1024**2
    print(f"HRI30_CS.npz: {size_mb:.1f} MB")
else:
    print("ERROR: HRI30_CS.npz not found at", npz_path)

# Verify checkpoint dir for output
ckpt_dir = '/content/drive/MyDrive/HRC_Research/checkpoints/ctrgcn_hri30_70_10_20'
os.makedirs(ckpt_dir, exist_ok=True)
print(f"Checkpoint dir ready: {ckpt_dir}")

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4
HRI30_CS.npz: 126.2 MB
Checkpoint dir ready: /content/drive/MyDrive/HRC_Research/checkpoints/ctrgcn_hri30_70_10_20


In [2]:
# Clone CTR-GCN repo and install dependencies
import os

repo_path = '/content/CTR-GCN'

if not os.path.exists(repo_path):
    !git clone https://github.com/Uason-Chen/CTR-GCN.git {repo_path}
else:
    print("Repo already exists.")

%cd /content/CTR-GCN

!pip install -q tensorboardX==2.1 torchpack fvcore iopath yacs thop
!pip install -q -e torchlight

print("Dependencies installed.")

Cloning into '/content/CTR-GCN'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 202 (delta 61), reused 48 (delta 48), pack-reused 102 (from 1)
Receiving objects: 100% (202/202), 1.67 MiB | 4.68 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/CTR-GCN
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.8/308.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.3/296.3 kB 29.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Dependencies installed.


In [3]:
# Apply all required patches to CTR-GCN
from pathlib import Path

# Patch 1: torchpack compatibility in torchlight/torchlight/util.py
util_path = Path('/content/CTR-GCN/torchlight/torchlight/util.py')
text = util_path.read_text()
if 'except ImportError' not in text:
    text = text.replace(
        'from torchpack.runner.hooks import PaviLogger',
        'try:\n    from torchpack.runner.hooks import PaviLogger\nexcept ImportError:\n    PaviLogger = None'
    )
    text = text.replace(
        'def log(self, *args, **kwargs):\n        try:',
        'def log(self, *args, **kwargs):\n        if PaviLogger is None:\n            return\n\n        try:'
    )
    util_path.write_text(text)
    print("Patch 1 applied: torchpack")
else:
    print("Patch 1 already applied.")

# Patch 2: tensorboardX dummy in main.py
main_path = Path('/content/CTR-GCN/main.py')
text2 = main_path.read_text()
if 'class SummaryWriter' not in text2:
    text2 = text2.replace(
        'from tensorboardX import SummaryWriter',
        '''try:
    from tensorboardX import SummaryWriter
except Exception:
    class SummaryWriter:
        def __init__(self, *args, **kwargs): pass
        def add_scalar(self, *args, **kwargs): pass
        def close(self): pass'''
    )
    main_path.write_text(text2)
    print("Patch 2 applied: tensorboardX dummy")
else:
    print("Patch 2 already applied.")

# Patch 3: yaml.safe_load in main.py
text3 = main_path.read_text()
if 'yaml.safe_load' not in text3:
    text3 = text3.replace('default_arg = yaml.load(f)', 'default_arg = yaml.safe_load(f)')
    main_path.write_text(text3)
    print("Patch 3 applied: yaml.safe_load")
else:
    print("Patch 3 already applied.")

print("All patches done.")

Patch 1 applied: torchpack
Patch 2 applied: tensorboardX dummy
Patch 3 applied: yaml.safe_load
All patches done.


In [4]:
# Download pretrained CTR-GCN weights
import os
import gdown

weights_dir = '/content/CTR-GCN/pretrained_model/NTU60_Xsub/CTRGCN_joint_89.9'
weights_path = os.path.join(weights_dir, 'runs-60-37560.pt')

os.makedirs(weights_dir, exist_ok=True)

if not os.path.exists(weights_path) or os.path.getsize(weights_path) < 1024*1024:
    print("Downloading pretrained CTR-GCN weights...")
    gdown.download(
        'https://drive.google.com/uc?id=1eVlsxaODkJ6Zhhwfauf7Q142tCzK1FvC',
        weights_path,
        quiet=False
    )
else:
    print("Weights already exist.")

size_mb = os.path.getsize(weights_path) / 1024**2
print(f"Weights size: {size_mb:.1f} MB")
assert size_mb > 4, "ERROR: weights file too small — download may have failed"
print("Weights verified OK.")

Downloading...
From: https://drive.google.com/uc?id=1eVlsxaODkJ6Zhhwfauf7Q142tCzK1FvC
To: /content/CTR-GCN/pretrained_model/NTU60_Xsub/CTRGCN_joint_89.9/runs-60-37560.pt
100%|██████████| 5.95M/5.95M [00:00<00:00, 32.2MB/s]

Weights size: 5.7 MB
Weights verified OK.


In [5]:
# Load HRI30 data and verify shapes
import numpy as np

npz_path = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/ctrgcn_format/HRI30_CS.npz'
data = np.load(npz_path, allow_pickle=True)

x_train = data['x_train']
y_train = data['y_train']
x_test  = data['x_test']
y_test  = data['y_test']

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)
print("x_train dtype:", x_train.dtype)
print("Label range train:", y_train.min(), "to", y_train.max())
print("Label range test:", y_test.min(), "to", y_test.max())
print("Unique train labels:", len(np.unique(y_train)))
print("Unique test labels:", len(np.unique(y_test)))

x_train shape: (2058, 3, 150, 25, 1)
y_train shape: (2058,)
x_test shape: (588, 3, 150, 25, 1)
y_test shape: (588,)
x_train dtype: float32
Label range train: 0 to 29
Label range test: 0 to 29
Unique train labels: 30
Unique test labels: 30


In [6]:
# Build CTR-GCN and load weights correctly
import sys
import torch
import torch.nn as nn
from collections import OrderedDict

sys.path.insert(0, '/content/CTR-GCN')
from model.ctrgcn import Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Build model with 30 classes (num_person=1 makes data_bn size 75)
model = Model(
    num_class=30,
    num_point=25,
    num_person=1,
    graph='graph.ntu_rgb_d.Graph',
    graph_args={'labeling_mode': 'spatial'}
)

# Load pretrained NTU60 weights (num_person=2 makes data_bn size 150)
weights_path = '/content/CTR-GCN/pretrained_model/NTU60_Xsub/CTRGCN_joint_89.9/runs-60-37560.pt'
pretrained = torch.load(weights_path, map_location='cpu', weights_only=False)

pretrained_filtered = OrderedDict()
for k, v in pretrained.items():
    if k.startswith('module.'):
        k = k[7:]
    if not k.startswith('fc.') and not k.startswith('data_bn.'):
        pretrained_filtered[k] = v

missing, unexpected = model.load_state_dict(pretrained_filtered, strict=False)
print(f"Missing keys (Should be 'fc' and 'data_bn' related): {missing}")

model = model.to(device)
print("\nModel loaded successfully on", device)

Using device: cuda
Missing keys (Should be 'fc' and 'data_bn' related): ['data_bn.weight', 'data_bn.bias', 'data_bn.running_mean', 'data_bn.running_var', 'fc.weight', 'fc.bias']

Model loaded successfully on cuda


In [7]:
# Create DataLoader
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# Convert to tensors — CTR-GCN expects (N, C, T, V, M)
x_train_t = torch.FloatTensor(x_train)
y_train_t = torch.LongTensor(y_train.astype(np.int64))
x_test_t  = torch.FloatTensor(x_test)
y_test_t  = torch.LongTensor(y_test.astype(np.int64))

train_dataset = TensorDataset(x_train_t, y_train_t)
test_dataset  = TensorDataset(x_test_t,  y_test_t)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")
print(f"Sample batch shape: {next(iter(train_loader))[0].shape}")

Train batches: 65 | Test batches: 19
Sample batch shape: torch.Size([32, 3, 150, 25, 1])


In [8]:
# PRE DIAGNOSTIC
print("============ DIAGNOSTIC START ============")
model.eval()
try:
    batch_x, batch_y = next(iter(train_loader))
    print(f"Loaded Batch Shape: {batch_x.shape}")

    if len(batch_x.shape) != 5:
        raise ValueError(f"CRITICAL: CTR-GCN expects 5 dimensions (N, C, T, V, M). Got {len(batch_x.shape)}")

    batch_x = batch_x.to(device)
    output = model(batch_x)
    print(f"Model Output Shape: {output.shape}")

    if output.shape[1] != 30:
         raise ValueError(f"CRITICAL: Expected 30 HRI30 classes, but model output {output.shape[1]}")

    print("Forward pass successful! No dimensional crashes.")
    print("STATUS: 100% SAFE TO PROCEED TO TRAINING!")

except Exception as e:
    print(f"\n CRITICAL ERROR DETECTED:")
    print(f"Type: {type(e).__name__}")
    print(f"Message: {e}")
    print("\nDO NOT proceed to Cell 8.")

============ DIAGNOSTIC START ============
Loaded Batch Shape: torch.Size([32, 3, 150, 25, 1])
Model Output Shape: torch.Size([32, 30])
Forward pass successful! No dimensional crashes.
STATUS: 100% SAFE TO PROCEED TO TRAINING!


In [9]:
# Fine-tune CTR-GCN on HRI30
import torch.optim as optim
import torch.nn as nn
import os

# UPDATED PATH for new weights
CHECKPOINT_DIR = '/content/drive/MyDrive/HRC_Research/checkpoints/ctrgcn_hri30_70_10_20'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Split Learning Rates to prevent catastrophic forgetting
backbone_params = [p for name, p in model.named_parameters() if 'fc' not in name]
head_params     = [p for name, p in model.named_parameters() if 'fc' in name]

optimizer = optim.SGD([
    {'params': backbone_params, 'lr': 1e-3},
    {'params': head_params,     'lr': 1e-2}
], momentum=0.9, weight_decay=0.0004, nesterov=True)

scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 40], gamma=0.1)
criterion = nn.CrossEntropyLoss()

best_acc   = 0.0
best_epoch = 0

for epoch in range(1, 51):
    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        _, predicted = output.max(1)
        train_correct += predicted.eq(batch_y).sum().item()
        train_total += batch_x.size(0)

    train_loss /= train_total
    train_acc = 100.0 * train_correct / train_total

    # --- Eval ---
    model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            output = model(batch_x)
            loss = criterion(output, batch_y)

            test_loss += loss.item() * batch_x.size(0)
            _, predicted = output.max(1)
            test_correct += predicted.eq(batch_y).sum().item()
            test_total += batch_x.size(0)

    test_loss /= test_total
    test_acc = 100.0 * test_correct / test_total
    scheduler.step()

    print(f"Epoch {epoch:02d}/50 | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Test Loss: {test_loss:.44} Acc: {test_acc:.2f}%")

    if epoch % 5 == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'test_acc': test_acc},
                   os.path.join(CHECKPOINT_DIR, f'epoch_{epoch:02d}.pt'))

    if test_acc > best_acc:
        best_acc = test_acc
        best_epoch = epoch
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'test_acc': test_acc},
                   os.path.join(CHECKPOINT_DIR, 'best_ctrgcn_hri30.pt'))
        print(f"  → New best model saved! Test Acc: {test_acc:.2f}%")

print(f"\nTraining complete. Best Test Accuracy: {best_acc:.2f}% at epoch {best_epoch}")

Epoch 01/50 | Train Loss: 2.8309 Acc: 19.73% | Test Loss: 1.8527970103179516314639840857125818729400635 Acc: 34.69%
  → New best model saved! Test Acc: 34.69%
Epoch 02/50 | Train Loss: 1.4604 Acc: 47.76% | Test Loss: 1.625788859769600636084874167863745242357254 Acc: 38.95%
  → New best model saved! Test Acc: 38.95%
Epoch 03/50 | Train Loss: 1.0647 Acc: 59.38% | Test Loss: 1.2196604986580050766775684678577817976474762 Acc: 48.64%
  → New best model saved! Test Acc: 48.64%
Epoch 04/50 | Train Loss: 0.8389 Acc: 68.95% | Test Loss: 1.0750243339408822595970605107140727341175079 Acc: 53.74%
  → New best model saved! Test Acc: 53.74%
Epoch 05/50 | Train Loss: 0.6688 Acc: 76.09% | Test Loss: 1.2210083980949557602713184678577817976474762 Acc: 49.49%
Epoch 06/50 | Train Loss: 0.5236 Acc: 84.16% | Test Loss: 0.97355936984626612051840766071109101176261902 Acc: 58.84%
  → New best model saved! Test Acc: 58.84%
Epoch 07/50 | Train Loss: 0.4361 Acc: 87.90% | Test Loss: 1.01664334576146142552488527144

In [10]:
## Per-class accuracy and save results
import numpy as np
import os
import torch

CHECKPOINT_DIR = '/content/drive/MyDrive/HRC_Research/checkpoints/ctrgcn_hri30_70_10_20'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best checkpoint
best_path = os.path.join(CHECKPOINT_DIR, 'best_ctrgcn_hri30.pt')
checkpoint = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        output  = model(batch_x)
        _, predicted = output.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

overall_acc = 100.0 * (all_preds == all_labels).mean()
print(f"Overall Test Accuracy (best model): {overall_acc:.2f}%")

print("\nPer-class accuracy:")
for c in range(30):
    mask = all_labels == c
    class_acc = 100.0 * (all_preds[mask] == all_labels[mask]).mean()
    print(f"  Class {c:02d}: {class_acc:.1f}%")

# Save to Drive
results_dir = '/content/drive/MyDrive/HRC_Research/results/accuracy_logs'
os.makedirs(results_dir, exist_ok=True)
# UPDATED FILENAME
results_path = os.path.join(results_dir, 'ctrgcn_hri30_70_10_20_results.txt')

with open(results_path, 'w') as f:
    f.write(f"CTR-GCN Fine-tuned on HRI30 (70/10/20 Split)\n")
    f.write(f"Best epoch: {checkpoint['epoch']}\n")
    f.write(f"Overall Test Accuracy: {overall_acc:.2f}%\n\n")
    f.write("Per-class accuracy:\n")
    for c in range(30):
        mask = all_labels == c
        class_acc = 100.0 * (all_preds[mask] == all_labels[mask]).mean()
        f.write(f"  Class {c:02d}: {class_acc:.1f}%\n")

print(f"\nResults saved to Drive: ctrgcn_hri30_70_10_20_results.txt")
print("\n=== PHASE 2.4 COMPLETE ===")

Overall Test Accuracy (best model): 67.69%

Per-class accuracy:
  Class 00: 63.2%
  Class 01: 60.0%
  Class 02: 65.0%
  Class 03: 85.0%
  Class 04: 68.4%
  Class 05: 60.0%
  Class 06: 85.0%
  Class 07: 50.0%
  Class 08: 47.4%
  Class 09: 73.7%
  Class 10: 40.0%
  Class 11: 80.0%
  Class 12: 78.9%
  Class 13: 60.0%
  Class 14: 50.0%
  Class 15: 73.7%
  Class 16: 73.7%
  Class 17: 94.7%
  Class 18: 80.0%
  Class 19: 40.0%
  Class 20: 65.0%
  Class 21: 60.0%
  Class 22: 75.0%
  Class 23: 89.5%
  Class 24: 68.4%
  Class 25: 75.0%
  Class 26: 95.0%
  Class 27: 63.2%
  Class 28: 52.6%
  Class 29: 60.0%

Results saved to Drive: ctrgcn_hri30_70_10_20_results.txt

=== PHASE 2.4 COMPLETE ===
